# Logistic Regression
**Course:** Foundations of Machine Learning  
**Instructor:** Sayan CHAKI, LIRIS (UMR 5205 CNRS), École Centrale de Lyon, Université Lumière Lyon 2, INSA Lyon

**Lab 2.** Sigmoid and log-loss, gradient descent and Newton (IRLS) from scratch, regularisation, feature expansion, ROC/PR, calibration, softmax regression.

> Run in Google Colab: *Runtime → Run all*. All datasets ship with scikit-learn, so no download is needed.


## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
np.random.seed(0)
plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score, log_loss)

## 1. The sigmoid and the log-loss
$\sigma(z) = 1/(1+e^{-z})$. We use a numerically stable implementation.

In [ ]:
def sigmoid(z):
    out = np.empty_like(z, dtype=float)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out

z = np.linspace(-8, 8, 200)
plt.plot(z, sigmoid(z), label="sigma(z)")
plt.plot(z, sigmoid(z) * (1 - sigmoid(z)), label="sigma'(z)")
plt.legend(); plt.title("Sigmoid and its derivative"); plt.show()

## 2. A 2D binary problem

In [ ]:
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=400, n_features=2, n_redundant=0, n_informative=2,
                           n_clusters_per_class=1, class_sep=1.2, flip_y=0.05, random_state=3)
X = StandardScaler().fit_transform(X)
Xb = np.c_[np.ones(len(X)), X]
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=15, edgecolor="k", lw=0.3)
plt.title("Data"); plt.show()

### 2.1 Gradient descent from scratch
$J(\mathbf w) = -\frac1n\sum_i [y_i\log p_i + (1-y_i)\log(1-p_i)] + \frac\lambda2\|\mathbf w_{1:}\|^2$, 
$\nabla J = \frac1n \mathbf X^\top(\mathbf p-\mathbf y) + \lambda \mathbf w_{1:}$.

In [ ]:
def bce(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def logreg_gd(X, y, lr=0.5, epochs=500, lam=0.0):
    w = np.zeros(X.shape[1]); hist = []
    for _ in range(epochs):
        p = sigmoid(X @ w)
        grad = X.T @ (p - y) / len(y)
        grad[1:] += lam * w[1:]              # do not penalise the bias
        w -= lr * grad
        hist.append(bce(y, p))
    return w, hist

w_gd, hist_gd = logreg_gd(Xb, y)
print("GD weights:", w_gd, " train acc:", accuracy_score(y, sigmoid(Xb @ w_gd) > 0.5))

### 2.2 Newton's method (IRLS)
$\mathbf w \leftarrow \mathbf w - (\mathbf X^\top \mathbf S \mathbf X)^{-1}\mathbf X^\top(\mathbf p - \mathbf y)$ with $\mathbf S=\mathrm{diag}(p_i(1-p_i))$. Note how few iterations it needs.

In [ ]:
def logreg_newton(X, y, iters=10, lam=1e-6):
    w = np.zeros(X.shape[1]); hist = []
    for _ in range(iters):
        p = sigmoid(X @ w)
        S = p * (1 - p)
        H = (X * S[:, None]).T @ X / len(y) + lam * np.eye(X.shape[1])
        g = X.T @ (p - y) / len(y) + lam * w
        w -= np.linalg.solve(H, g)
        hist.append(bce(y, sigmoid(X @ w)))
    return w, hist

w_nt, hist_nt = logreg_newton(Xb, y)
plt.plot(hist_gd, label="gradient descent")
plt.plot(np.arange(1, len(hist_nt) + 1), hist_nt, "o-", label="Newton")
plt.xscale("log"); plt.xlabel("iteration"); plt.ylabel("log-loss"); plt.legend(); plt.show()

sk = LogisticRegression(C=1e6).fit(X, y)
print("Newton :", w_nt)
print("sklearn:", np.r_[sk.intercept_, sk.coef_.ravel()])

### 2.3 Decision boundary and probability map

In [ ]:
def plot_proba(predict_proba, X, y, ax=None, title=""):
    ax = ax or plt.gca()
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min()-1, X[:, 0].max()+1, 300),
                         np.linspace(X[:, 1].min()-1, X[:, 1].max()+1, 300))
    P = predict_proba(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    cs = ax.contourf(xx, yy, P, levels=20, cmap="coolwarm", alpha=0.6)
    ax.contour(xx, yy, P, levels=[0.5], colors="k")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=12, edgecolor="k", lw=0.3)
    ax.set_title(title)
    return cs

plot_proba(lambda Z: sigmoid(np.c_[np.ones(len(Z)), Z] @ w_nt), X, y, title="P(y=1 | x), Newton")
plt.show()

### 2.4 Effect of regularisation ($C = 1/\lambda$)
Small $C$: strong regularisation, soft transitions. Large $C$: sharp, confident boundary.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, Cval in zip(axes, [0.01, 1, 100]):
    m = LogisticRegression(C=Cval).fit(X, y)
    plot_proba(lambda Z: m.predict_proba(Z)[:, 1], X, y, ax=ax, title=f"C={Cval}, ||w||={np.linalg.norm(m.coef_):.2f}")
plt.show()

## 3. Nonlinear boundaries with feature expansion

In [ ]:
from sklearn.datasets import make_circles
from sklearn.preprocessing import PolynomialFeatures
Xc, yc = make_circles(n_samples=400, noise=0.1, factor=0.5, random_state=0)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
lin = LogisticRegression().fit(Xc, yc)
poly = make_pipeline(PolynomialFeatures(2), StandardScaler(), LogisticRegression()).fit(Xc, yc)
plot_proba(lambda Z: lin.predict_proba(Z)[:, 1], Xc, yc, ax=axes[0], title=f"linear, acc={lin.score(Xc, yc):.2f}")
plot_proba(lambda Z: poly.predict_proba(Z)[:, 1], Xc, yc, ax=axes[1], title=f"degree 2, acc={poly.score(Xc, yc):.2f}")
plt.show()

## 4. Real data: breast cancer diagnosis
569 tumours, 30 features, target malignant (0) / benign (1).

In [ ]:
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(bc.data, bc.target, stratify=bc.target, test_size=0.25, random_state=0)

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
grid = GridSearchCV(pipe, {"logisticregression__C": np.logspace(-3, 3, 13)}, cv=5, scoring="roc_auc")
grid.fit(Xtr, ytr)
print("best C:", grid.best_params_, " CV AUC:", round(grid.best_score_, 4))

proba = grid.predict_proba(Xte)[:, 1]
pred = grid.predict(Xte)
print("test accuracy:", accuracy_score(yte, pred), " test log-loss:", round(log_loss(yte, proba), 4))
print(classification_report(yte, pred, target_names=bc.target_names))
ConfusionMatrixDisplay(confusion_matrix(yte, pred), display_labels=bc.target_names).plot(cmap="Blues")
plt.grid(False); plt.show()

### 4.1 ROC and precision-recall curves

In [ ]:
fpr, tpr, thr = roc_curve(yte, proba)
prec, rec, _ = precision_recall_curve(yte, proba)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(fpr, tpr, label=f"AUC = {roc_auc_score(yte, proba):.3f}"); ax[0].plot([0, 1], [0, 1], "k--")
ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].set_title("ROC"); ax[0].legend()
ax[1].plot(rec, prec, label=f"AP = {average_precision_score(yte, proba):.3f}")
ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision"); ax[1].set_title("Precision-Recall"); ax[1].legend()
plt.show()

### 4.2 Changing the decision threshold

In [ ]:
for tau in [0.3, 0.5, 0.7, 0.9]:
    p = (proba >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(yte, p).ravel()
    print(f"tau={tau:.1f}  precision={tp/(tp+fp):.3f}  recall={tp/(tp+fn):.3f}  FP={fp} FN={fn}")

### 4.3 Interpreting coefficients (standardised features) and L1 sparsity

In [ ]:
coef = grid.best_estimator_.named_steps["logisticregression"].coef_.ravel()
order = np.argsort(np.abs(coef))[::-1][:10]
plt.barh(np.array(bc.feature_names)[order][::-1], coef[order][::-1])
plt.title("Top-10 coefficients (L2)"); plt.show()

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  # sklearn>=1.8 prefers l1_ratio; penalty="l1" works across versions
for Cval in [0.01, 0.1, 1]:
    l1 = make_pipeline(StandardScaler(), LogisticRegression(penalty="l1", solver="liblinear", C=Cval)).fit(Xtr, ytr)
    nz = np.sum(l1[-1].coef_ != 0)
    print(f"L1, C={Cval}: {nz} non-zero coefficients, test acc={l1.score(Xte, yte):.3f}")

### 4.4 Calibration

In [ ]:
from sklearn.calibration import calibration_curve
frac_pos, mean_pred = calibration_curve(yte, proba, n_bins=8)
plt.plot(mean_pred, frac_pos, "o-", label="logistic regression")
plt.plot([0, 1], [0, 1], "k--", label="perfect")
plt.xlabel("mean predicted probability"); plt.ylabel("fraction of positives"); plt.legend(); plt.show()

## 5. Multiclass: softmax regression on handwritten digits
$p(y=k\mid\mathbf x) = \frac{\exp(\mathbf w_k^\top\mathbf x)}{\sum_j \exp(\mathbf w_j^\top\mathbf x)}$, gradient $\frac1n\mathbf X^\top(\mathbf P - \mathbf Y)$.

In [ ]:
from sklearn.datasets import load_digits
digits = load_digits()
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(digits.data, digits.target, test_size=0.25,
                                              stratify=digits.target, random_state=0)
sc = StandardScaler().fit(Xd_tr)
Xd_tr_s, Xd_te_s = sc.transform(Xd_tr), sc.transform(Xd_te)

def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)      # log-sum-exp trick
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

def softmax_regression(X, y, K, lr=0.5, epochs=300, lam=1e-3):
    Xb = np.c_[np.ones(len(X)), X]
    W = np.zeros((Xb.shape[1], K))
    Y = np.eye(K)[y]
    for _ in range(epochs):
        P = softmax(Xb @ W)
        G = Xb.T @ (P - Y) / len(y)
        G[1:] += lam * W[1:]
        W -= lr * G
    return W

W = softmax_regression(Xd_tr_s, yd_tr, K=10)
pred_d = softmax(np.c_[np.ones(len(Xd_te_s)), Xd_te_s] @ W).argmax(1)
print("from-scratch softmax test acc:", accuracy_score(yd_te, pred_d))
sk_mc = LogisticRegression(max_iter=2000).fit(Xd_tr_s, yd_tr)
print("sklearn multinomial test acc :", sk_mc.score(Xd_te_s, yd_te))

fig, axes = plt.subplots(1, 10, figsize=(14, 1.8))
for k, ax in enumerate(axes):
    ax.imshow(W[1:, k].reshape(8, 8), cmap="RdBu"); ax.set_title(str(k)); ax.axis("off")
plt.suptitle("Learned weight templates per class"); plt.show()

## 6. Exercises
1. Plot the log-loss and the 0-1 loss as a function of $\|\mathbf w\|$ on a **linearly separable** dataset without regularisation. What happens and why?
2. Implement mini-batch SGD for the logistic loss and compare convergence with GD and Newton.
3. On the breast cancer data, find the threshold that guarantees recall $\ge 0.99$ for the malignant class. What precision do you get?
4. Compare one-vs-rest and multinomial logistic regression on digits (`sklearn.multiclass.OneVsRestClassifier`).

In [ ]:
# Your code here